# Welcome to OpenModelStudio

Your workspace is pre-configured with the **`openmodelstudio` SDK**. Everything is auto-connected — your project, auth token, and API endpoint are already set.

This notebook walks through a **complete end-to-end ML workflow**:

**Load data → Engineer features → Store hyperparams → Register model → Train → Infer → Experiment tracking**

Run each cell one by one.

---
## Cell 1 — Imports

In [ ]:
import openmodelstudio
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

---
## Cell 2 — Load and prep data

In [ ]:
df = openmodelstudio.load_dataset("titanic")
df = df.dropna(subset=["Survived", "Pclass", "Age", "Fare"])
print(f"Loaded {len(df)} rows")
df.head()

---
## Cell 3 — Register features in the Feature Store

In [ ]:
openmodelstudio.create_features(df,
    feature_names=["Pclass", "Age", "Fare"],
    group_name="titanic-v1",
    transforms={"Age": "standard_scaler", "Fare": "min_max_scaler"})

df_scaled = openmodelstudio.load_features("titanic-v1", df=df)
print("Features registered and transforms applied")
df_scaled[["Pclass", "Age", "Fare"]].describe()

---
## Cell 4 — Store hyperparameters

In [ ]:
openmodelstudio.create_hyperparameters("rf-tuned", {
    "n_estimators": 200,
    "max_depth": 8,
    "min_samples_split": 4,
    "random_state": 42,
})
print("Hyperparameters stored")

---
## Cell 5 — Register model (no local training needed)

In [ ]:
params = openmodelstudio.load_hyperparameters("rf-tuned")
clf = RandomForestClassifier(**params)

handle = openmodelstudio.register_model("titanic-rf", model=clf)
print(handle)
# ModelHandle(id='...', name='titanic-rf', version=1)

---
## Cell 6 — Train through the system

In [ ]:
job = openmodelstudio.start_training(handle.model_id,
    hyperparameter_set="rf-tuned",
    wait=True)

print(f"Training status: {job['status']}")
# Check the Jobs page in the UI for live loss/accuracy charts

---
## Cell 7 — View training logs

In [ ]:
logs = openmodelstudio.get_logs(job["id"])
for entry in logs[-10:]:  # last 10 entries
    print(f"[{entry['level']}] {entry['message']}")

---
## Cell 8 — Run inference through the system

In [ ]:
result = openmodelstudio.start_inference(handle.model_id,
    input_data={"features": [[3, 25.0, 7.25], [1, 38.0, 71.28]]},
    wait=True)

print(f"Status: {result['status']}")
print(f"Predictions: {result.get('metrics')}")

---
## Cell 9 — Create an experiment and record the run

In [ ]:
exp = openmodelstudio.create_experiment("titanic-tuning",
    description="Comparing RF hyperparameter configs")

openmodelstudio.add_experiment_run(exp["id"],
    job_id=job["id"],
    parameters={"n_estimators": 200, "max_depth": 8, "min_samples_split": 4},
    metrics={"accuracy": 0.94})

print(f"Experiment: {exp['id']}")

---
## Cell 10 — Run a second config and add to experiment

In [ ]:
openmodelstudio.create_hyperparameters("rf-deep", {
    "n_estimators": 500,
    "max_depth": 15,
    "min_samples_split": 2,
    "random_state": 42,
})

clf2 = RandomForestClassifier(**openmodelstudio.load_hyperparameters("rf-deep"))
handle2 = openmodelstudio.register_model("titanic-rf", model=clf2)

job2 = openmodelstudio.start_training(handle2.model_id,
    hyperparameter_set="rf-deep",
    wait=True)

openmodelstudio.add_experiment_run(exp["id"],
    job_id=job2["id"],
    parameters={"n_estimators": 500, "max_depth": 15, "min_samples_split": 2},
    metrics={"accuracy": 0.96})

print(f"Second run recorded — status: {job2['status']}")

---
## Cell 11 — Compare experiment runs

In [ ]:
comparison = openmodelstudio.compare_experiment_runs(exp["id"])
runs = openmodelstudio.list_experiment_runs(exp["id"])

print(f"\n{'Run':<6} {'n_estimators':<14} {'max_depth':<11} {'accuracy':<10} {'status'}")
print("-" * 55)
for i, r in enumerate(runs):
    p = r.get("parameters", {})
    m = r.get("metrics", {})
    print(f"#{i+1:<5} {str(p.get('n_estimators','')):<14} {str(p.get('max_depth','')):<11} {str(m.get('accuracy','')):<10} {r.get('status', '')}")

---
## Cell 12 — Monitor all jobs

In [ ]:
jobs = openmodelstudio.list_jobs()
print(f"\n{'Job ID':<10} {'Type':<12} {'Status':<12} {'Model'}")
print("-" * 55)
for j in jobs[:10]:
    print(f"{j['id'][:8]:<10} {j['job_type']:<12} {j['status']:<12} {j.get('model_id', '')[:8]}")

---
## Cell 13 — Load trained model back into notebook (optional)

In [ ]:
clf_loaded = openmodelstudio.load_model("titanic-rf")
print(f"Model type: {type(clf_loaded).__name__}")
print(f"Estimators: {clf_loaded.n_estimators}")

---

## SDK Reference

The SDK reads these environment variables automatically (set by the workspace pod):
- `OPENMODELSTUDIO_API_URL` — API endpoint
- `OPENMODELSTUDIO_TOKEN` — Auth token
- `OPENMODELSTUDIO_WORKSPACE_ID` — Current workspace
- `OPENMODELSTUDIO_PROJECT_ID` — Current project

See the full SDK docs at [sdk/python/README.md](../sdk/python/README.md).